# AKD-EXT Closed Loop CM1 Playground


## Closed loop workflow - CM1

A pipeline of 4 agents for end-to-end research workflows. These agents are designed to work sequentially, where each stage's output feeds into the next.

**Required env vars:** `OPENAI_API_KEY`

| Stage | Agent | Purpose |
|-------|-------|---------|
| 2 | CapabilityFeasibilityMapperAgent | Assesses feasibility of a research question |
| 3 | WorkflowSpecBuilderAgent | Builds experiment workflow spec from hypotheses |
| 4 | ExperimentImplementationAgent | Creates implementation workspace (SLURM, configs) |
| 5 | ResearchReportGeneratorAgent | Generates analysis notebooks and reports |

### Stage 1: Gap Agent Output

In [7]:
# Stage 1: Gap Agent Output

research_question = """# Environmental Stability Control via Input Sounding

## Research Question

**How does atmospheric stability in the environmental temperature profile (stable vs unstable input sounding) influence tropical cyclone intensity and structural evolution?**

---

## Problem Framing

Many idealized tropical cyclone simulations assume a single environmental thermodynamic profile, but the vertical temperature stability of the environment has a strong influence on convection organization, boundary-layer overturning, and eyewall dynamics. Most previous work has focused on surface fluxes or drag processes, with the role of background thermodynamic stability (controlled via the input sounding) rarely isolated.

Understanding whether instability-driven convection enhances storm intensification — compared to stable stratification that suppresses vertical motion — can help clarify the thermodynamic constraints on tropical cyclone development.

- **Origin:** Research design extension
- **Confidence:** Moderate
- **Rationale:** Environmental stability directly controls convective buoyancy, vertical mass flux, and eyewall convection strength — key drivers of tropical cyclone intensity.

---

## Evidence Anchor (Intra-Corpus)

- **Convective instability and tropical cyclone intensification:** Emanuel (1986, 1995)
- **Environmental thermodynamic control of convection:** Rotunno & Emanuel (1987)
- **Role of environmental stability in eyewall convection and storm structure:** Montgomery & Smith (2014)

---

## Variables / Diagnostics

- Maximum wind speed evolution
- Minimum central pressure
- Convective intensity and vertical velocity distribution
- Eyewall structure and radius of maximum wind (RMW)
- CAPE / atmospheric stability metrics derived from sounding
- Boundary layer thermodynamic structure
- Eyewall updraft magnitude and organization

---

## Causality Guardrails

- Stability changes **must only be introduced** through temperature profile modifications in the input sounding.
- Surface fluxes and drag formulations **must remain constant** to isolate thermodynamic effects.
- Interpretation should focus on structural and dynamical response before attributing causal mechanisms.

---

## Hypothesis

**Hypothesis 3.1:**
Tropical cyclone intensity will increase more rapidly under an unstable environmental temperature profile because enhanced buoyancy strengthens eyewall convection and vertical mass flux, whereas a stable temperature profile suppresses deep convection, leading to weaker storm intensification and reduced peak intensity.

---

## Why This Remains Secondary

Environmental stability interacts strongly with other thermodynamic controls such as surface fluxes, moisture availability, and drag processes. Without isolating these mechanisms first, interpretation of stability-driven intensity changes may be confounded by coupled processes.
"""

### Stage 2: Capability/Feasibility Mapper Agent

In [8]:
from akd_ext.agents import (
    CapabilityFeasibilityMapperAgent,
    CapabilityFeasibilityMapperConfig,
    CapabilityFeasibilityMapperInputSchema,
)

In [11]:
# Stage 2: Capability Feasibility Mapper
mapper_agent = CapabilityFeasibilityMapperAgent(CapabilityFeasibilityMapperConfig())

result = await mapper_agent.arun(
    CapabilityFeasibilityMapperInputSchema(
        research_questions_md=research_question,
    )
)

print(result.report)

# CARE Capability & Feasibility Mapping Report — Environmental Stability Control via Input Sounding (CM1)

## Disclaimer (Human Decision Boundary)
This report provides capability/feasibility assessments and evidence paths only. It does not make a final decision to run experiments; human approval is required.

---

## Step 1 — Validate Inputs
### Inputs received
- Research questions/hypotheses markdown: **present** (user-provided)
- CM1 configuration corpus: **present** (multiple example cases with `namelist.input` and some `input_sounding` files)
- Cluster IT context (UAH “Matrix”): **present** (queues, limits, pre-emption notes)

### Critical missing inputs (affecting confidence)
- No CM1 source-code or full namelist-reference documentation beyond the provided config corpus (limits ability to confirm some physics/diagnostic semantics beyond parameter presence).
- No stated **queue walltime limits** for Matrix partitions (affects cluster-fit assessment for long 3D runs).

Evidence path

In [12]:
# Saving the md file for testing purpose
from pathlib import Path

report_md_path = Path("result_report.md")
report_md_path.write_text(result.report, encoding="utf-8")
print(f"Saved report to: {report_md_path}")
print(result.report)

Saved report to: result_report.md
# CARE Capability & Feasibility Mapping Report — Environmental Stability Control via Input Sounding (CM1)

## Disclaimer (Human Decision Boundary)
This report provides capability/feasibility assessments and evidence paths only. It does not make a final decision to run experiments; human approval is required.

---

## Step 1 — Validate Inputs
### Inputs received
- Research questions/hypotheses markdown: **present** (user-provided)
- CM1 configuration corpus: **present** (multiple example cases with `namelist.input` and some `input_sounding` files)
- Cluster IT context (UAH “Matrix”): **present** (queues, limits, pre-emption notes)

### Critical missing inputs (affecting confidence)
- No CM1 source-code or full namelist-reference documentation beyond the provided config corpus (limits ability to confirm some physics/diagnostic semantics beyond parameter presence).
- No stated **queue walltime limits** for Matrix partitions (affects cluster-fit assessment

In [14]:
# read the md file for testing purpose
from pathlib import Path

report_md_path = Path("result_report.md")
report_md_content = report_md_path.read_text(encoding="utf-8")

### Stage 3: Workflow Spec Builder

In [15]:
from akd_ext.agents import (
    WorkflowSpecBuilderAgent,
    WorkflowSpecBuilderConfig,
    WorkflowSpecBuilderInputSchema,
)

In [16]:
# Stage 3: Workflow Spec Builder
# Takes Gap Agent output (hypotheses) + Stage 2 feasibility report
spec_agent = WorkflowSpecBuilderAgent(WorkflowSpecBuilderConfig())

spec_result = await spec_agent.arun(
    WorkflowSpecBuilderInputSchema(
        stage_1_hypotheses=research_question,
        stage_2_feasibility=report_md_content,
    )
)

print("SPEC:")
print("=" * 80)
print(spec_result.spec)
print("\nREASONING:")
print("=" * 80)
print(spec_result.reasoning)

SPEC:
# Metadata
- status: draft
- model: CM1
- workflow_name: Environmental stability sensitivity via input_sounding (axisymmetric hurricane)
- rq_tag_or_rq_id: RQ-001
- hypothesis_id: H-001 (Hypothesis 3.1)
- experiment_tag: STAB
- baseline_experiment_id: EXP_STAB_baseline
- planned_experiment_count: 3
- baseline_case_source: run/config_files/hurricane_axisymmetric
- created_date: 2026-03-31

## Sources
- run/config_files/cpm_RadConvEquil/namelist.input
- run/config_files/hurricane_axisymmetric/input_sounding
- run/config_files/hurricane_axisymmetric/namelist.input

# Control Definition
## Baseline intent
Establish a control tropical cyclone simulation using the existing CM1 idealized **axisymmetric hurricane** configuration, then isolate environmental stability effects by modifying **only** the environmental temperature (implemented as potential temperature, `theta`) in `input_sounding` while holding surface fluxes, drag, and all other physics identical.

## Baseline configuration r

In [17]:
# Save the workflow spec as a markdown file
spec_md_path = Path("workflow_spec.md")
spec_md_path.write_text(spec_result.spec + "\n\n---\n\n# Reasoning\n\n" + spec_result.reasoning, encoding="utf-8")
print(f"Saved workflow spec to: {spec_md_path}")

Saved workflow spec to: workflow_spec.md


In [1]:
from pathlib import Path

workflow_spec_path = Path("workflow_spec.md")
workflow_spec_content = workflow_spec_path.read_text(encoding="utf-8")

### Stage 4 A: Experiment Implementation Agent - LLM Part to generate the JSON

In [11]:
from akd_ext.agents import (
    ExperimentImplementationAgent,
    ExperimentImplementationConfig,
    ExperimentImplementationInputSchema,
    ExperimentImplementationOutputSchema,
)

In [12]:
# Stage 4A: Experiment Implementation Planner
# GPT generates experiments, calls job_submit MCP, returns job_id + report
impl_agent = ExperimentImplementationAgent(ExperimentImplementationConfig())

impl_result = await impl_agent.arun(
    ExperimentImplementationInputSchema(stage_3_spec=workflow_spec_content)
)

if isinstance(impl_result, ExperimentImplementationOutputSchema):
    print(f"Job ID: {impl_result.job_id}")
    print()
    print("REPORT:")
    print("=" * 80)
    print(impl_result.report)
else:
    print(impl_result.content)

Job ID: 0ed13ebd

REPORT:
## 1. Job Submission
- **job_id:** `0ed13ebd`
- **workspace_name:** `cm1_STAB_env_stability_axisym_hurr`
- **base_template:** `hurricane_axisymmetric` (source: `run/config_files/hurricane_axisymmetric`)

## 2. Experiments Implemented (Total: 3)

### EXP_STAB_baseline (baseline)
- **Edits:** none (uses template files as-is)
- **Notes:** Leaves `&param12` surface flux/drag settings unchanged by construction; leaves `isnd=7` unchanged.

### EXP_STAB_001 (more stable; +2 K theta perturbation)
- **Target file:** `input_sounding`
- **Edits (theta column only):**
  1. `z=1500–4000 m`: add **+2 K** with **linear_ramp** (0 → +2)
  2. `z=4000–10000 m`: add **+2 K** constant
  3. `z=10000–12000 m`: add **+2 K** constant
  4. `z=10000–12000 m`: add **-2 K** with **linear_ramp** (0 → -2)
     - Combined with (3), this produces a **linear taper from +2 K at 10 km down to 0 K at 12 km**.
- **Feasibility flag:** `CONSTRAINT_DEPENDENT` (per Stage 3: requires `isnd=7` to remain

In [13]:
# Save the implementation report
if isinstance(impl_result, ExperimentImplementationOutputSchema):
    impl_md_path = Path("experiment_implementation.md")
    impl_md_path.write_text(impl_result.report, encoding="utf-8")
    print(f"Saved implementation report to: {impl_md_path}")
    print(f"Job ID: {impl_result.job_id}")

Saved implementation report to: experiment_implementation.md
Job ID: 0ed13ebd


### Stage 5: Research Report Generator

Takes the Stage 3 workflow spec + experiment IDs from Stage 4A.
The agent checks experiment status and fetches figure URLs via MCP tools,
then generates a publication-style scientific report.

**Input:** Workflow spec + experiment IDs (from Stage 4A)

**Output:** Markdown report with Abstract, Introduction, Methodology, Results, Discussion, Conclusion

**Note:** Without the MCP server, use `tools=[]` and override the prompt to skip status/figure checks.

In [14]:
from akd_ext.agents import (
    ResearchReportGeneratorAgent,
    ResearchReportGeneratorConfig,
    ResearchReportGeneratorInputSchema,
    ResearchReportGeneratorOutputSchema,
)

In [15]:
experiment_ids = "EXP_stability_baseline,EXP_stability_001,EXP_stability_002"

In [16]:
# Stage 5: Research Report Generator
# Uses job_id from Stage 4A to check status and fetch figures via MCP
job_id = impl_result.job_id
print(f"Job ID: {job_id}")

report_agent = ResearchReportGeneratorAgent(ResearchReportGeneratorConfig())

report_result = await report_agent.arun(
    ResearchReportGeneratorInputSchema(
        workflow_spec=workflow_spec_content,
        job_id=job_id,
    ),
)

print("RESEARCH REPORT:")
print("=" * 80)
if isinstance(report_result, ResearchReportGeneratorOutputSchema):
    print(report_result.report)
else:
    print(report_result.content)

Job ID: 0ed13ebd
RESEARCH REPORT:
## 1. Abstract
Environmental thermodynamic stability has been hypothesised to modulate tropical cyclone (TC) intensification by regulating deep convective buoyancy in the eyewall. An idealised CM1 axisymmetric hurricane configuration was used as a controlled framework in which environmental stability was perturbed exclusively by modifying the potential temperature (θ) profile in the file-based `input_sounding` while holding surface fluxes, drag, and physics settings fixed. Three experiments were conducted: a baseline control, a “more stable” mid–upper-tropospheric θ perturbation (+2 K with vertical tapering between 1.5–12 km), and a “more unstable” perturbation (−2 K with the same tapering). Visual inspection of the resulting intensity/structure diagnostics suggests stability-dependent modulation of intensification behaviour consistent with the stated hypothesis (pending researcher validation).

*This report was generated with AI assistance and require

In [17]:
# Save the research report
if isinstance(report_result, ResearchReportGeneratorOutputSchema):
    research_report_path = Path("research_report.md")
    research_report_path.write_text(report_result.report, encoding="utf-8")
    print(f"Saved research report to: {research_report_path}")
else:
    print("No structured report to save (agent returned TextOutput)")

Saved research report to: research_report.md
